# Guide: Risk, Stress, and Hedge Action

Runs portfolio shock propagation, decomposition, tail risk, and hedge optimization with deterministic synthetic returns.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().resolve().parent))

from src.risk.scenarios.em_scenarios import em_scenario_library
from src.risk.portfolio_shocks import Trade, propagate_scenario, decompose_pnl
from src.risk.hedging_optimizer import optimize_hedges, OptimizerConfig
from src.risk.tail_risk import historical_var, expected_shortfall


In [ ]:
portfolio = [
    Trade('T1','FRA',12_000_000,'front',dv01=-2800),
    Trade('T2','Swap',18_000_000,'belly',dv01=-4600),
    Trade('T3','XCCY_BasisSwap',9_000_000,'back',basis01=-2000,hedge_overlay=True),
    Trade('T4','FX_Forward',7_000_000,'front',fx_delta=-3_200_000,hedge_overlay=True),
]

scenarios = {s.name:s for s in em_scenario_library()}
base = propagate_scenario(portfolio, scenarios['capital_outflow_shock'])
stress = propagate_scenario(portfolio, scenarios['currency_devaluation_shock'])

base_total = base['pnl_total'].sum()
stress_total = stress['pnl_total'].sum()
pnl_compare = pd.DataFrame({'state':['base','stress'],'pnl_total':[base_total,stress_total]})
pnl_compare


In [ ]:
# Hedge optimization under stressed exposures.
exposure = np.array([120.0, -70.0, 35.0])
hedge_matrix = np.array([
    [-0.9,-0.2,-0.3],
    [0.3,-0.8,-0.2],
    [-0.2,-0.3,-0.7],
])
carry = np.array([0.5,0.25,0.35])
liq = np.array([0.45,0.30,0.95])

opt = optimize_hedges(
    exposure_vector=exposure,
    hedge_matrix=hedge_matrix,
    carry_vector=carry,
    liquidity_vector=liq,
    instruments=['IRS','XCCY','FXFWD'],
    tenor_bucket=['front','belly','front'],
    config=OptimizerConfig(max_notional=1.0),
)
opt['solution']


In [ ]:
# Deterministic synthetic daily P&L for tail metrics.
np.random.seed(404)
returns = pd.Series(np.random.normal(-0.0002, 0.004, 500))
var99 = historical_var(returns, 0.99)
es99 = expected_shortfall(returns, 0.99, method='historical')

dec = decompose_pnl(stress)
risk_snapshot = {
    'stress_total_pnl': float(stress_total),
    'delta_stress_minus_base': float(stress_total-base_total),
    'historical_var_99': float(var99),
    'historical_es_99': float(es99),
}
risk_snapshot


In [ ]:
class RiskStressExplainer:
    def __init__(self, pnl_compare: pd.DataFrame, risk_snapshot: dict, hedge_solution: pd.DataFrame):
        self.pnl_compare = pnl_compare
        self.risk_snapshot = risk_snapshot
        self.hedge_solution = hedge_solution

    def render_full_markdown(self) -> str:
        delta = self.risk_snapshot['delta_stress_minus_base']
        var99 = self.risk_snapshot['historical_var_99']
        es99 = self.risk_snapshot['historical_es_99']
        xccy = float(self.hedge_solution.loc[self.hedge_solution['instrument']=='XCCY','optimal_notional'].item())
        return f"""
### Interpretation (P&L / DV01 / Convexity / Basis / Hedge Action)
- **Macro shock change:** devaluation stress worsens total P&L by about **{delta:,.0f}** vs base shock.
- **DV01:** front/belly rate shocks dominate first-order losses through FRA/swap DV01.
- **Convexity:** tail metrics (VaR/ES) indicate nonlinear downside beyond first-order sensitivity.
- **Basis:** xccy basis shock is a key incremental driver in overlay positions.
- **Tail risk:** historical VaR(99%) ≈ **{var99:.4f}**, ES(99%) ≈ **{es99:.4f}** for synthetic returns.
- **Hedge action:** optimizer suggests XCCY hedge notional around **{xccy:.3f}** (sign indicates pay/receive direction); rebalance with FX forward hedge for devaluation scenarios.
""".strip()

explainer = RiskStressExplainer(pnl_compare, risk_snapshot, opt['solution'])
print(explainer.render_full_markdown())
